# SumCheck Protocol in JAX — Beginner-Friendly Tutorial

> **ECE-9413 Assignment 2 companion notebook**  
> This notebook walks you through every concept and every line of code you need, from scratch — no prior knowledge of zero-knowledge proofs required.

---

## What you will learn

| Section | What you build |
|---|---|
| 1. Intuition | Why SumCheck exists and what it solves |
| 2. Polynomials & Tables | How multilinear polynomials are stored as arrays |
| 3. Modular Arithmetic | 32-bit mod-add, mod-sub, mod-mul in JAX |
| 4. MLE Update | The core interpolation primitive |
| 5. One Round of SumCheck | Computing round evaluations g_i(0), g_i(1), ... |
| 6. Full SumCheck Loop | Putting all rounds together |
| 7. Composed Polynomials | Handling a*b, a*b+c, a*b*c |
| 8. GPU/TPU Optimisation | Which ops scale, which don't, and how to fix them |
| 9. Benchmarking | Measuring real throughput with JAX JIT |

---
## Section 0 — Setup

Run this cell first. It installs JAX and checks your hardware.

In [ ]:
# Install JAX with GPU support (Colab already has CUDA, so this just ensures the right wheel)
# If you're on CPU-only, change 'cuda12' -> 'cpu'
!pip install -q --upgrade "jax[cuda12]" jaxlib

import jax
import jax.numpy as jnp

# Enable 64-bit support (needed for large moduli later)
jax.config.update("jax_enable_x64", True)

print("JAX version:", jax.__version__)
print("Devices available:", jax.devices())
print("Default backend:", jax.default_backend())

---
## Section 1 — What Is SumCheck and Why Do We Care?

### The core problem

Imagine you have a huge multivariate polynomial `f(x1, x2, ..., xn)` and someone claims:

```
S = sum of f over every (x1,...,xn) in {0,1}^n
```

Verifying this naively means evaluating `f` at all `2^n` binary points and summing them — that is **exponential work**.

SumCheck lets a **verifier** check this claim by talking to a **prover** through `n` interactive rounds, doing only `O(n * d)` work (where `d` is the polynomial degree). The prover does the heavy lifting; the verifier just checks consistency.

### Why is this useful?

- Zero-knowledge proof systems (zkSNARKs, zkSTARKs) use SumCheck as a core sub-protocol.
- The papers cited in the assignment (NoCap, zkSpeed, zkPHIRE) all accelerate SumCheck on hardware.
- Understanding it gives you intuition for how modern privacy-preserving computation works.

### The roles

| Party | Job |
|---|---|
| **Prover** | Runs the heavy computation. Sends round polynomials g_i. **This is what you implement.** |
| **Verifier** | Checks consistency, sends random challenges. Provided by the harness. |

---
## Section 2 — Polynomials as Lookup Tables

### Multilinear polynomials

A polynomial is **multilinear** if every variable appears with degree at most 1.  
Examples that ARE multilinear:
- `x1 * x2 + x3`
- `x1 + 2*x2 + 4*x3`

Examples that are NOT multilinear:
- `x1^2 + x2`  (x1 has degree 2)
- `x1 * x2 * x2`  (x2 appears twice)

### Key insight: a table is enough

A multilinear polynomial over `n` variables is completely determined by its values at all `2^n` binary points `{0,1}^n`. So instead of storing symbolic coefficients, we store a flat array of length `2^n`.

Index `i` into the table corresponds to the assignment where bit `k` of `i` gives `x_{k+1}`. In this assignment, **x1 is the least-significant bit**.

In [ ]:
import jax.numpy as jnp

# -----------------------------------------------------------
# Example: f(x1, x2, x3) = x1 + 2*x2 + 4*x3  (mod 17)
#   - x1 is the least-significant bit (bit 0 of the index)
#   - x2 is bit 1
#   - x3 is bit 2 (most-significant bit)
# -----------------------------------------------------------

Q = 17  # our tiny prime modulus for this section

# Build the table manually from the formula
f_table = []
for idx in range(8):  # 2^3 = 8 entries
    x1 = (idx >> 0) & 1   # bit 0
    x2 = (idx >> 1) & 1   # bit 1
    x3 = (idx >> 2) & 1   # bit 2
    val = (x1 + 2*x2 + 4*x3) % Q
    f_table.append(val)
    print(f"  idx={idx}  (x3={x3}, x2={x2}, x1={x1})  f={val}")

f_table = jnp.array(f_table, dtype=jnp.uint32)
print("\nFull table as JAX array:", f_table)
print("Sum of all entries (= S):", int(jnp.sum(f_table)) % Q, "  (should be 11 mod 17)")

### Expression representation in this assignment

The assignment uses `list[list[str]]` to encode polynomial expressions:

```python
[['a']]           # just  a
[['a', 'b']]      # a*b
[['a', 'b'], ['c']]   # a*b + c
[['a', 'b', 'c']]     # a*b*c
```

Each variable (`a`, `b`, `c`, ...) has its own evaluation table stored in a dict called `eval_tables`.

In [ ]:
import jax.numpy as jnp

# Suppose we have a 3-variable example (2^3 = 8 entries per table)
# Let's make simple tables for demonstration
Q = 17
n = 3  # number of variables

eval_tables = {
    'a': jnp.array([0, 1, 2, 3, 4, 5, 6, 7], dtype=jnp.uint32),
    'b': jnp.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=jnp.uint32),
    'c': jnp.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=jnp.uint32),
}

# Expression:  a*b + c
expression = [['a', 'b'], ['c']]

# Compute f(x) = a(x)*b(x) + c(x) pointwise, then sum
f_vals = (eval_tables['a'] * eval_tables['b'] + eval_tables['c']) % Q
print("f = a*b + c at each index:", f_vals)
print("Claimed sum S:", int(jnp.sum(f_vals)) % Q)

---
## Section 3 — Modular Arithmetic Primitives (32-bit)

All SumCheck arithmetic is done **modulo a prime `q`** to keep values in a finite field. We need three primitives: add, subtract, multiply.

### Why modular arithmetic is tricky with fixed-width integers

- `uint32` can hold values up to ~4.3 billion.
- `a * b` where both are up to `q-1 ≈ 4.29e9` can produce a value up to `~1.84e19` — this **overflows** a 32-bit integer.
- The standard trick: widen to 64-bit for the multiply, then reduce mod q back to 32-bit.

### GPU/TPU note on modular arithmetic
These are **elementwise ops** — perfect for GPU/TPU. Every entry in the array is independent, so the hardware can process all `2^n` elements in parallel. This is the single biggest win from GPU acceleration.

In [ ]:
import jax
import jax.numpy as jnp

# ---------------------------------------------------------------------------
# mod_add_32: (a + b) mod q
#
# Addition of two uint32 values can exceed uint32 range only if q is close
# to 2^32. The safe way: cast to uint64, add, reduce, cast back.
# ---------------------------------------------------------------------------
def mod_add_32(a, b, q):
    # Cast to 64-bit to avoid overflow during the intermediate sum
    a64 = a.astype(jnp.uint64)
    b64 = b.astype(jnp.uint64)
    q64 = jnp.uint64(q)
    result = (a64 + b64) % q64
    return result.astype(jnp.uint32)


# ---------------------------------------------------------------------------
# mod_sub_32: (a - b) mod q
#
# Subtraction can go negative. Add q before subtracting to stay positive,
# then reduce.
# ---------------------------------------------------------------------------
def mod_sub_32(a, b, q):
    a64 = a.astype(jnp.uint64)
    b64 = b.astype(jnp.uint64)
    q64 = jnp.uint64(q)
    # Adding q ensures we never go negative in unsigned arithmetic
    result = (a64 + q64 - b64) % q64
    return result.astype(jnp.uint32)


# ---------------------------------------------------------------------------
# mod_mul_32: (a * b) mod q
#
# Multiplication is where 32-bit definitely overflows.
# Max value of a or b = q-1 ≈ 4.29e9.
# (q-1)^2 ≈ 1.84e19 which overflows uint32 (max ~4.29e9) and even uint64
# (max ~1.84e19). So we must use uint64 carefully.
# The max prime for 32-bit is 4_294_967_291 (just under 2^32).
# (4_294_967_291 - 1)^2 = ~1.844e19 which just fits in uint64 (max = 2^64-1).
# ---------------------------------------------------------------------------
def mod_mul_32(a, b, q):
    a64 = a.astype(jnp.uint64)
    b64 = b.astype(jnp.uint64)
    q64 = jnp.uint64(q)
    result = (a64 * b64) % q64
    return result.astype(jnp.uint32)


# ---------------------------------------------------------------------------
# Quick tests
# ---------------------------------------------------------------------------
Q = jnp.uint32(17)

a = jnp.array([5, 10, 16], dtype=jnp.uint32)
b = jnp.array([3,  9,  1], dtype=jnp.uint32)

print("mod_add_32([5,10,16], [3,9,1], 17) =", mod_add_32(a, b, Q))  # [8, 2, 0]
print("mod_sub_32([5,10,16], [3,9,1], 17) =", mod_sub_32(a, b, Q))  # [2, 1, 15]
print("mod_mul_32([5,10,16], [3,9,1], 17) =", mod_mul_32(a, b, Q))  # [15, 5, 16]

# Verify manually: 5+3=8, 10+9=19%17=2, 16+1=17%17=0
#                  5-3=2, 10-9=1, 16-1=15
#                  5*3=15, 10*9=90%17=5 (90=5*17+5), 16*1=16

### Vectorised vs scalar — the GPU advantage

Notice these functions work on **entire JAX arrays at once** (not element-by-element in a Python loop). When JAX runs on a GPU, all `2^n` elements are processed in parallel across thousands of CUDA cores. On a CPU with 8 cores, only 8 elements can run at once.

In [ ]:
import time

Q_big = jnp.uint32(4_294_967_291)  # the largest 32-bit prime

# Simulate a vars20 table: 2^20 = 1_048_576 entries
key = jax.random.PRNGKey(42)
big_a = jax.random.randint(key, shape=(2**20,), minval=0, maxval=int(Q_big), dtype=jnp.uint32)
big_b = jax.random.randint(key, shape=(2**20,), minval=0, maxval=int(Q_big), dtype=jnp.uint32)

# JIT-compile the multiply so we measure steady-state time (not compile time)
jit_mul = jax.jit(mod_mul_32)
_ = jit_mul(big_a, big_b, Q_big).block_until_ready()  # warm-up

t0 = time.perf_counter()
for _ in range(10):
    result = jit_mul(big_a, big_b, Q_big).block_until_ready()
t1 = time.perf_counter()

print(f"2^20 mod_mul_32 throughput: {10 / (t1 - t0):.1f} calls/sec")
print(f"Elements per second: {10 * 2**20 / (t1 - t0):.2e}")
print(f"Running on: {result.device}")

---
## Section 4 — The MLE Update: Interpolating to a New Point

After each SumCheck round, the verifier samples a random challenge `r`. We need to "fold" our table: replace each pair of entries `(f at x_i=0, f at x_i=1)` with a single entry `f at x_i=r`.

This is linear interpolation between the two boolean values:

```
mle_update(z, o, t) = (o - z) * t + z   (mod q)
```

Where:
- `z` = value when the variable is 0  (zero-entry, even index)
- `o` = value when the variable is 1  (one-entry, odd index)
- `t` = the random challenge we're interpolating to

### Why this works
This is just the slope-intercept form `y = m*x + b` for two points `(0, z)` and `(1, o)`. The slope is `o - z`, intercept is `z`. Evaluating at `t` gives us `(o - z)*t + z`.

### Table halving
After one MLE update, the table shrinks from size `2^n` to `2^(n-1)` because each pair of entries collapses to one.

In [ ]:
import jax.numpy as jnp

def mle_update_32(zero_eval, one_eval, target_eval, *, q):
    """
    Compute (one_eval - zero_eval) * target_eval + zero_eval  (mod q)
    for every pair in the table simultaneously.

    Args:
        zero_eval : array of shape (N//2,) — values at x_i = 0 (even indices)
        one_eval  : array of shape (N//2,) — values at x_i = 1 (odd indices)
        target_eval : scalar — the challenge r to interpolate to
        q         : the prime modulus (scalar)

    Returns:
        array of shape (N//2,) — the folded table
    """
    diff = mod_sub_32(one_eval, zero_eval, q)           # (o - z) mod q
    scaled = mod_mul_32(diff, target_eval, q)            # (o - z) * t mod q
    result = mod_add_32(scaled, zero_eval, q)            # + z, mod q
    return result


# -----------------------------------------------------------
# Worked example from the assignment intro doc:
# Table = [0, 1, 2, 3, 4, 5, 6, 7], Q=17, r1=8
#
# Pairs: (0,1), (2,3), (4,5), (6,7)
#   (1-0)*8+0 = 8
#   (3-2)*8+2 = 10
#   (5-4)*8+4 = 12
#   (7-6)*8+6 = 14
# Expected output: [8, 10, 12, 14]
# -----------------------------------------------------------
Q = jnp.uint32(17)
table = jnp.array([0, 1, 2, 3, 4, 5, 6, 7], dtype=jnp.uint32)
r1    = jnp.uint32(8)

# Split into even (x1=0) and odd (x1=1) entries
evens = table[0::2]  # indices 0, 2, 4, 6  ->  values [0, 2, 4, 6]
odds  = table[1::2]  # indices 1, 3, 5, 7  ->  values [1, 3, 5, 7]

print("Even entries (x1=0):", evens)
print("Odd  entries (x1=1):", odds)

folded = mle_update_32(evens, odds, r1, q=Q)
print("Folded table after r1=8:", folded)  # expect [8, 10, 12, 14]

# One more fold: apply r2=5 to get the final 1-entry table
r2 = jnp.uint32(5)
evens2 = folded[0::2]  # [8, 12]
odds2  = folded[1::2]  # [10, 14]
folded2 = mle_update_32(evens2, odds2, r2, q=Q)
print("Folded again after r2=5:", folded2)  # expect [1, 5]

---
## Section 5 — One Round of SumCheck

In each round `i`, the prover must send evaluations of the round polynomial `g_i(t)` at `t = 0, 1, ..., d` where `d` is the degree of the composed expression.

### Degree determines how many evaluation points you need

| Expression | Degree | Eval points needed |
|---|---|---|
| `a` | 1 | g(0), g(1) |
| `a*b` | 2 | g(0), g(1), g(2) |
| `a*b + c` | 2 | g(0), g(1), g(2) |
| `a*b*c` | 3 | g(0), g(1), g(2), g(3) |

### How to compute g_i(t) for a given t

For each pair of even/odd entries in the table:
1. Use MLE update to get the value at `x_i = t` for each polynomial variable.
2. Compute the composed expression (e.g., `a*b + c`) at that point.
3. Accumulate across all pairs.

For `t=0`: just take even entries directly (MLE update with t=0 gives back the even entry).  
For `t=1`: just take odd entries directly.  
For `t=2, 3, ...`: use MLE update to extrapolate.

In [ ]:
import jax
import jax.numpy as jnp


def compute_expression(tables_at_point, expression, q):
    """
    Evaluate the composed expression pointwise.

    tables_at_point: dict mapping variable name -> array of values (one per pair)
    expression: list[list[str]], e.g. [['a','b'], ['c']]
    q: prime modulus (uint32 scalar)

    Returns: array of values (sum of all additive terms, each being product of factors)
    """
    total = None
    for term in expression:
        # Multiply all factors in this additive term
        term_val = None
        for var in term:
            v = tables_at_point[var]
            if term_val is None:
                term_val = v
            else:
                term_val = mod_mul_32(term_val, v, q)
        # Add this term to the running total
        if total is None:
            total = term_val
        else:
            total = mod_add_32(total, term_val, q)
    return total


def compute_round_evals(eval_tables, expression, q):
    """
    Compute the round polynomial evaluations g(0), g(1), ..., g(degree).

    For each evaluation point t = 0, 1, ..., degree:
      - For t=0: use even entries of each variable's table.
      - For t=1: use odd entries of each variable's table.
      - For t>=2: use MLE update to extrapolate from (even, odd) to t.

    Then sum the composed expression values across all pairs.
    """
    # Determine the degree = total number of variable factors in the expression
    degree = sum(len(term) for term in expression)
    num_eval_points = degree + 1  # need d+1 points to define a degree-d polynomial

    # Get the number of pairs (half the table size)
    any_table = next(iter(eval_tables.values()))
    n = len(any_table)
    assert n % 2 == 0, "Table length must be even (we always split into pairs)"

    # Pre-split each variable's table into even/odd halves
    evens = {var: tbl[0::2] for var, tbl in eval_tables.items()}
    odds  = {var: tbl[1::2] for var, tbl in eval_tables.items()}

    round_evals = []
    for t_int in range(num_eval_points):
        t = jnp.array(t_int, dtype=jnp.uint32)  # evaluation point as a JAX scalar

        if t_int == 0:
            # At t=0, MLE_update(z, o, 0) = z  (just the even entries)
            tables_at_t = evens
        elif t_int == 1:
            # At t=1, MLE_update(z, o, 1) = o  (just the odd entries)
            tables_at_t = odds
        else:
            # For t >= 2, extrapolate using MLE update
            tables_at_t = {
                var: mle_update_32(evens[var], odds[var], t, q=q)
                for var in eval_tables
            }

        # Evaluate the expression at this t across all pairs, then sum
        vals_at_t = compute_expression(tables_at_t, expression, q)
        g_t = jnp.sum(vals_at_t.astype(jnp.uint64)) % jnp.uint64(q)
        round_evals.append(int(g_t))

    return round_evals


# -----------------------------------------------------------
# Test with the 3-variable example, expression = [['a']] (just 'a')
# Table: [0, 1, 2, 3, 4, 5, 6, 7], Q=17
# Round 1:
#   g(0) = sum of even entries = 0+2+4+6 = 12
#   g(1) = sum of odd  entries = 1+3+5+7 = 16
# -----------------------------------------------------------
Q = jnp.uint32(17)
eval_tables_demo = {
    'a': jnp.array([0, 1, 2, 3, 4, 5, 6, 7], dtype=jnp.uint32),
}
expression_a = [['a']]

evals = compute_round_evals(eval_tables_demo, expression_a, Q)
print(f"Round evals for 'a': {evals}")
print(f"Check: g(0)+g(1) = {evals[0]}+{evals[1]} = {(evals[0]+evals[1]) % 17} (should be 11 = S mod 17)")

---
## Section 6 — The Full SumCheck Loop

Now we put all the rounds together. The loop:

1. Round 0: compute initial claim `S = sum of f over all {0,1}^n`.
2. For each round `i = 1..n`:
   a. Compute round evaluations `g_i(0), g_i(1), ..., g_i(d)`.
   b. Apply the MLE update to all tables using challenge `r_i` → tables halve.
3. Return `(claim0, all_round_evals)` as JAX arrays.

**Important constraint from the assignment:** rounds **must be serialised** — you cannot parallelise across rounds because each round's tables depend on the previous round's challenge.

In [ ]:
import jax
import jax.numpy as jnp


def apply_mle_update_to_all_tables(eval_tables, challenge, q):
    """
    Apply mle_update to every variable's table using the given challenge.
    Each table shrinks by half.
    """
    new_tables = {}
    for var, tbl in eval_tables.items():
        evens = tbl[0::2]
        odds  = tbl[1::2]
        new_tables[var] = mle_update_32(evens, odds, challenge, q=q)
    return new_tables


def sumcheck_32(eval_tables, *, q, expression, challenges, num_rounds):
    """
    Full SumCheck prover (32-bit).

    Args:
        eval_tables : dict[str -> jnp.ndarray of uint32]  shape (2^num_rounds,)
        q           : uint32 scalar — prime modulus
        expression  : list[list[str]] — the composed polynomial
        challenges  : 1D jnp.ndarray of uint32, length = num_rounds - 1
                      (the final challenge is verifier-only, not passed here)
        num_rounds  : int — number of SumCheck rounds (= number of variables)

    Returns:
        (claim0, round_evals)
        - claim0     : scalar JAX array  — initial claimed sum S
        - round_evals: 2D JAX array of shape (num_rounds, degree+1)
    """
    degree = sum(len(term) for term in expression)
    num_eval_points = degree + 1

    # --- Step 1: Compute claim0 = sum of f over all Boolean inputs ---
    # Evaluate the expression pointwise, then sum all entries.
    # This is just compute_expression on the full table, then reduce-sum.
    f_vals = compute_expression(eval_tables, expression, q)
    claim0 = jnp.sum(f_vals.astype(jnp.uint64)) % jnp.uint64(q)
    claim0 = claim0.astype(jnp.uint32)

    # --- Step 2: Run each round ---
    current_tables = eval_tables  # will shrink each round
    all_round_evals = []          # collect (d+1) evals per round

    for round_idx in range(num_rounds):
        # 2a. Compute the round polynomial evaluations g(0), g(1), ..., g(d)
        round_evals = compute_round_evals(current_tables, expression, q)
        all_round_evals.append(round_evals)

        # 2b. Apply MLE update using the challenge for this round.
        # We skip the update in the last round because the last challenge is
        # kept by the verifier only (not passed to us).
        if round_idx < len(challenges):
            r = challenges[round_idx]  # scalar uint32
            current_tables = apply_mle_update_to_all_tables(current_tables, r, q)

    # Stack round_evals into a 2D array: shape (num_rounds, degree+1)
    round_evals_array = jnp.array(all_round_evals, dtype=jnp.uint32)

    return claim0, round_evals_array


# -----------------------------------------------------------
# Verify on the 3-variable example from the intro doc:
#   f(x1,x2,x3) = x1 + 2*x2 + 4*x3  (mod 17)
#   Table = [0,1,2,3,4,5,6,7]
#   Challenges: r1=8, r2=5  (r3=11 is verifier-only, not passed)
#
# Expected:
#   claim0 = 11
#   Round 1: g1(0)=12, g1(1)=16
#   Round 2: g2(0)=3,  g2(1)=7
#   Round 3: g3(0)=1,  g3(1)=5
# -----------------------------------------------------------
Q = jnp.uint32(17)
tables_demo = {
    'a': jnp.array([0, 1, 2, 3, 4, 5, 6, 7], dtype=jnp.uint32)
}
challenges_demo = jnp.array([8, 5], dtype=jnp.uint32)  # r1=8, r2=5; r3=11 is verifier-only

claim0, round_evals = sumcheck_32(
    tables_demo,
    q=Q,
    expression=[['a']],
    challenges=challenges_demo,
    num_rounds=3,
)

print(f"claim0 = {int(claim0)}  (expected 11)")
for i, row in enumerate(round_evals):
    print(f"Round {i+1}: g({', '.join(str(t) for t in range(len(row)))}) = {[int(v) for v in row]}")

print("\nExpected:")
print("Round 1: g(0,1) = [12, 16]")
print("Round 2: g(0,1) = [3, 7]")
print("Round 3: g(0,1) = [1, 5]")

---
## Section 7 — Composed Polynomials: a\*b, a\*b+c, a\*b\*c

Now let's test with the actual expressions the assignment grades you on. The key difference from the single-polynomial case:

- Each variable (`a`, `b`, `c`, ...) has its **own separate table**.
- The composition (e.g., `a*b`) is applied **pointwise** before summing.
- For degree > 1, you need extra evaluation points (at `t=2`, `t=3`, ...) obtained via MLE extrapolation.

The `compute_expression` and `compute_round_evals` functions we wrote above already handle this correctly.

In [ ]:
import jax.numpy as jnp

# Small 2-variable example (2^2 = 4 entries per table) to check each expression
Q = jnp.uint32(17)

# Define tables for a, b, c over {0,1}^2
#   idx 0: (x2=0, x1=0)
#   idx 1: (x2=0, x1=1)
#   idx 2: (x2=1, x1=0)
#   idx 3: (x2=1, x1=1)
tables_2var = {
    'a': jnp.array([1, 2, 3, 4], dtype=jnp.uint32),
    'b': jnp.array([2, 3, 1, 2], dtype=jnp.uint32),
    'c': jnp.array([0, 1, 2, 3], dtype=jnp.uint32),
}
challenges_2var = jnp.array([7], dtype=jnp.uint32)  # only 1 challenge (num_rounds - 1)

expressions_to_test = {
    'a':       [['a']],
    'a*b':     [['a', 'b']],
    'a*b + c': [['a', 'b'], ['c']],
    'a*b*c':   [['a', 'b', 'c']],
}

print("Tables:")
for var, tbl in tables_2var.items():
    print(f"  {var}: {tbl.tolist()}")
print()

for expr_name, expression in expressions_to_test.items():
    degree = sum(len(term) for term in expression)

    # Compute true sum for verification
    a = tables_2var['a']
    b = tables_2var['b']
    c = tables_2var['c']
    if expr_name == 'a':
        true_sum = int(jnp.sum(a.astype(jnp.uint64))) % 17
    elif expr_name == 'a*b':
        true_sum = int(jnp.sum((a.astype(jnp.uint64)*b.astype(jnp.uint64)) % 17)) % 17
    elif expr_name == 'a*b + c':
        true_sum = int(jnp.sum((a.astype(jnp.uint64)*b.astype(jnp.uint64) % 17 + c.astype(jnp.uint64)) % 17)) % 17
    elif expr_name == 'a*b*c':
        true_sum = int(jnp.sum((a.astype(jnp.uint64)*b.astype(jnp.uint64) % 17 * c.astype(jnp.uint64)) % 17)) % 17

    claim0, round_evals = sumcheck_32(
        tables_2var,
        q=Q,
        expression=expression,
        challenges=challenges_2var,
        num_rounds=2,
    )

    match = "OK" if int(claim0) == true_sum else f"MISMATCH (true={true_sum})"
    print(f"Expression '{expr_name}' (degree {degree}):")
    print(f"  claim0 = {int(claim0)}  [{match}]")
    print(f"  round_evals per round: {round_evals.tolist()}")
    # Consistency check: g(0) + g(1) mod q should equal claim0
    r0_evals = round_evals[0].tolist()
    consistency = (r0_evals[0] + r0_evals[1]) % 17
    print(f"  Consistency check: g1(0)+g1(1) = {r0_evals[0]}+{r0_evals[1]} = {consistency}  "
          f"[{'OK' if consistency == int(claim0) else 'FAIL'}]")
    print()

---
## Section 8 — GPU and TPU Optimisation

This is where hardware really matters. Let's analyse every operation and categorise it.

### 8.1 — Mental model: what GPUs/TPUs are good at

| Hardware | Strength | Weakness |
|---|---|---|
| CPU | Sequential logic, branching, small arrays | Massively parallel elementwise ops |
| GPU | Elementwise ops on large arrays, reductions | Sequential loops, small arrays |
| TPU | Matrix multiplications, batched ops | Irregular memory access patterns |

### 8.2 — Operation-by-operation analysis

| Operation | Parallelisable? | GPU/TPU benefit | Notes |
|---|---|---|---|
| `mod_add / mod_sub / mod_mul` on full table | **YES** | Large | Elementwise, perfectly parallel |
| `jnp.sum` (reduce-sum for claim0) | **YES** | Large | GPU parallel reduction |
| `table[0::2]` / `table[1::2]` (split) | **YES** | Large | Contiguous memory strided access |
| MLE update on all pairs simultaneously | **YES** | Large | Elementwise on half-table |
| For-loop over `t = 0..degree` | **NO** | None | Must be sequential; degree is small (1–3) |
| For-loop over `round_idx = 0..n` | **NO** | None | Rounds depend on previous challenge |
| `compute_expression` inner loop over terms | **Partial** | Medium | Small loop, but each step is parallel over entries |

### 8.3 — Key optimisation 1: `jax.jit`

JAX's JIT compiler traces your Python code and compiles it to XLA, which:
- Fuses multiple elementwise ops into a single GPU kernel (no intermediate arrays written to memory).
- Automatically maps operations to all available GPU cores.
- Eliminates Python interpreter overhead.

In [ ]:
import jax
import jax.numpy as jnp
import time

Q_big = jnp.uint32(4_294_967_291)
key = jax.random.PRNGKey(0)
N = 2**20  # 1M entries (vars20 table size)

big_table = jax.random.randint(key, (N,), 0, int(Q_big), dtype=jnp.uint32)
big_b     = jax.random.randint(key, (N,), 0, int(Q_big), dtype=jnp.uint32)

# --- Without JIT ---
def unjitted_op(a, b, q):
    return mod_mul_32(a, b, q)

# --- With JIT ---
jitted_op = jax.jit(mod_mul_32)

# Warm-up JIT
_ = jitted_op(big_table, big_b, Q_big).block_until_ready()

RUNS = 20

t0 = time.perf_counter()
for _ in range(RUNS):
    unjitted_op(big_table, big_b, Q_big).block_until_ready()
t_nojit = (time.perf_counter() - t0) / RUNS * 1000

t0 = time.perf_counter()
for _ in range(RUNS):
    jitted_op(big_table, big_b, Q_big).block_until_ready()
t_jit = (time.perf_counter() - t0) / RUNS * 1000

print(f"Without JIT: {t_nojit:.2f} ms/call")
print(f"With    JIT: {t_jit:.2f} ms/call")
print(f"Speedup from JIT: {t_nojit/t_jit:.1f}x")
print(f"Backend: {jax.default_backend()}")

### 8.4 — Key optimisation 2: Batch the degree-loop with `jax.vmap`

The inner loop `for t in range(degree+1)` in `compute_round_evals` runs `d+1` times (usually 2, 3, or 4 times). We can replace this with `jax.vmap` to vectorise across all `t` values simultaneously.

**Before:** compute g(0), g(1), g(2) sequentially  
**After:** compute g(0), g(1), g(2) all at once by mapping over a `t` array

> Note: this matters more on TPUs (which love batched matrix ops) than on GPUs (where the kernel launch overhead for 2–3 sequential calls is small). For large `n` it still helps by reducing Python overhead.

In [ ]:
import jax
import jax.numpy as jnp

def compute_g_at_t(t_val, evens, odds, expression, q):
    """
    Compute the composed expression summed over all pairs,
    where each variable is interpolated to the given t_val.

    t_val: scalar uint32
    evens / odds: dict of arrays
    Returns: scalar uint32 — g(t_val)
    """
    t = jnp.uint32(t_val)
    # Build tables_at_t by MLE-updating to t
    tables_at_t = {
        var: mle_update_32(evens[var], odds[var], t, q=q)
        for var in evens
    }
    vals = compute_expression(tables_at_t, expression, q)
    return jnp.sum(vals.astype(jnp.uint64)) % jnp.uint64(q)


def compute_round_evals_vmapped(eval_tables, expression, q):
    """
    Vectorised version: compute all g(t) values for t in [0, ..., degree]
    using a single vectorised call instead of a Python for-loop.
    """
    degree = sum(len(term) for term in expression)
    t_values = jnp.arange(degree + 1, dtype=jnp.uint32)  # [0, 1, 2, ...]

    evens = {var: tbl[0::2] for var, tbl in eval_tables.items()}
    odds  = {var: tbl[1::2] for var, tbl in eval_tables.items()}

    # vmap maps compute_g_at_t over the first arg (t_val), holding the rest constant
    vmapped_g = jax.vmap(
        lambda t: compute_g_at_t(t, evens, odds, expression, q)
    )
    return vmapped_g(t_values)  # returns array of shape (degree+1,)


# Verify it gives the same results as the sequential version
Q = jnp.uint32(17)
tables_verify = {
    'a': jnp.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=jnp.uint32),
    'b': jnp.array([2, 3, 1, 2, 3, 1, 2, 3], dtype=jnp.uint32),
}
expr = [['a', 'b']]  # degree 2

sequential_evals = compute_round_evals(tables_verify, expr, Q)
vmapped_evals    = compute_round_evals_vmapped(tables_verify, expr, Q)

print("Sequential:", sequential_evals)
print("Vmapped:   ", vmapped_evals.tolist())
print("Match:", sequential_evals == vmapped_evals.tolist())

### 8.5 — Key optimisation 3: Avoid Python dicts inside JIT — use stacked arrays

A Python `dict` of JAX arrays prevents JAX from tracing through it cleanly. If you JIT a function that does `for var in dict: ...`, Python's loop runs at trace-time (not ideal for large expressions).

A better approach: **stack all variable tables into a single 2D array** of shape `(num_vars, table_size)`. Then matrix/vectorised ops replace the inner Python loops entirely.

In [ ]:
import jax
import jax.numpy as jnp

def tables_to_stack(eval_tables, var_order):
    """Stack dict of tables into a 2D array: shape (num_vars, table_size)."""
    return jnp.stack([eval_tables[v] for v in var_order], axis=0)


def compute_expression_stacked(stacked_tables, term_indices, q):
    """
    Evaluate expression on stacked_tables without a Python dict or loop.

    stacked_tables: shape (num_vars, half_table_size) — already split (evens or odds)
    term_indices: list of lists of int — which row index in stacked_tables each factor uses
                  e.g. for 'a*b + c' with var_order=['a','b','c']: [[0,1], [2]]
    q: uint32 scalar

    This removes the Python dict lookup from inside JIT-compiled code.
    """
    total = None
    for term in term_indices:
        term_val = stacked_tables[term[0]]
        for idx in term[1:]:
            term_val = mod_mul_32(term_val, stacked_tables[idx], q)
        total = term_val if total is None else mod_add_32(total, term_val, q)
    return total


# Example: expression = a*b + c,  var_order = ['a', 'b', 'c']
var_order = ['a', 'b', 'c']
Q = jnp.uint32(17)
tables_3 = {
    'a': jnp.array([1, 2, 3, 4], dtype=jnp.uint32),
    'b': jnp.array([2, 3, 1, 2], dtype=jnp.uint32),
    'c': jnp.array([0, 1, 2, 3], dtype=jnp.uint32),
}

# term_indices for 'a*b + c': a=index 0, b=index 1, c=index 2
term_indices = [[0, 1], [2]]  # matches [['a','b'], ['c']]

stacked = tables_to_stack(tables_3, var_order)
print("Stacked shape:", stacked.shape)  # (3, 4)

result = compute_expression_stacked(stacked, term_indices, Q)
expected = (tables_3['a'] * tables_3['b'] + tables_3['c']) % 17
print("Stacked result:", result.tolist())
print("Expected:      ", expected.tolist())
print("Match:", result.tolist() == expected.tolist())

### 8.6 — Key optimisation 4: JIT the entire round

The biggest win is to JIT-compile the entire inner computation of a round (split + MLE update + expression evaluation + reduction) into a single XLA program. This way:
- No Python overhead between operations.
- XLA can fuse all elementwise kernels.
- Memory traffic is minimised (intermediate buffers stay in GPU registers/SRAM).

You cannot JIT the outer `for round_idx` loop because JAX doesn't know the shape of the tables at trace time (they shrink each round). But you CAN JIT everything inside one round.

In [ ]:
import jax
import jax.numpy as jnp
import functools
import time


@functools.partial(jax.jit, static_argnames=['expression_tuple'])
def jit_round_evals(stacked_tables, q, expression_tuple):
    """
    JIT-compiled computation of one round's g(0), g(1), ..., g(d).

    expression_tuple: tuple[tuple[int,...]] — term_indices as a static argument
                      (static because JAX needs to trace through the Python loop)
    stacked_tables: shape (num_vars, table_size)
    """
    # Split evens/odds along the last axis
    evens = stacked_tables[:, 0::2]  # shape (num_vars, table_size//2)
    odds  = stacked_tables[:, 1::2]  # shape (num_vars, table_size//2)

    degree = sum(len(term) for term in expression_tuple)
    num_eval_points = degree + 1

    round_evals = []
    for t_int in range(num_eval_points):
        t = jnp.uint32(t_int)
        if t_int == 0:
            tables_at_t = evens
        elif t_int == 1:
            tables_at_t = odds
        else:
            # MLE update applied to every row simultaneously
            diff    = mod_sub_32(odds, evens, q)        # shape (num_vars, half)
            scaled  = mod_mul_32(diff, t, q)             # broadcast scalar t
            tables_at_t = mod_add_32(scaled, evens, q)  # shape (num_vars, half)

        # Evaluate expression using term_indices
        vals = compute_expression_stacked(tables_at_t, list(expression_tuple), q)
        g_t = jnp.sum(vals.astype(jnp.uint64)) % jnp.uint64(q)
        round_evals.append(g_t.astype(jnp.uint32))

    return jnp.stack(round_evals)  # shape (degree+1,)


# Benchmark JIT round vs non-JIT round on a large table
Q_big = jnp.uint32(4_294_967_291)
key = jax.random.PRNGKey(1)
N = 2**20  # vars20
num_vars = 3

big_stacked = jax.random.randint(key, (num_vars, N), 0, int(Q_big), dtype=jnp.uint32)

# expression a*b*c, term_indices = ((0,1,2),)
expr_tuple = ((0, 1, 2),)

# Warm up
_ = jit_round_evals(big_stacked, Q_big, expr_tuple).block_until_ready()

RUNS = 10
t0 = time.perf_counter()
for _ in range(RUNS):
    jit_round_evals(big_stacked, Q_big, expr_tuple).block_until_ready()
elapsed = (time.perf_counter() - t0) / RUNS * 1000

print(f"JIT round (vars20, a*b*c): {elapsed:.2f} ms/round")
print(f"Backend: {jax.default_backend()}")

### 8.7 — GPU vs CPU comparison

Run this cell to see side-by-side timing. The key operation is `mod_mul_32` on a `2^20` array — the most compute-intensive part of each round.

In [ ]:
import jax
import jax.numpy as jnp
import time

Q_big = jnp.uint32(4_294_967_291)
key = jax.random.PRNGKey(99)
N = 2**20

all_devices = jax.devices()
print("All available devices:", all_devices)

jit_mul = jax.jit(mod_mul_32)

for device in all_devices:
    a = jax.device_put(jax.random.randint(key, (N,), 0, int(Q_big), dtype=jnp.uint32), device)
    b = jax.device_put(jax.random.randint(key, (N,), 0, int(Q_big), dtype=jnp.uint32), device)
    q = jax.device_put(Q_big, device)

    # Warm-up
    _ = jit_mul(a, b, q).block_until_ready()

    RUNS = 20
    t0 = time.perf_counter()
    for _ in range(RUNS):
        jit_mul(a, b, q).block_until_ready()
    elapsed_ms = (time.perf_counter() - t0) / RUNS * 1000

    throughput = N / (elapsed_ms / 1000) / 1e9
    print(f"{str(device):30s}  {elapsed_ms:.3f} ms/call  ({throughput:.2f} Gelem/s)")

### 8.8 — Summary: what to optimise and how

```
┌─────────────────────────────────────────────────────────────────────┐
│                   OPTIMISATION GUIDE                                │
├───────────────────────┬───────────────────────┬─────────────────────┤
│  Operation            │  How to optimise       │  Where it helps     │
├───────────────────────┼───────────────────────┼─────────────────────┤
│ mod_mul, mod_add,     │ Already vectorised;    │ GPU: huge win       │
│ mod_sub on full table │ wrap in jax.jit        │ TPU: large win      │
├───────────────────────┼───────────────────────┼─────────────────────┤
│ Loop over degree      │ Use jax.vmap over t    │ GPU: modest         │
│ (t=0,1,...,d)         │ or unroll statically   │ TPU: meaningful     │
├───────────────────────┼───────────────────────┼─────────────────────┤
│ Dict lookup in JIT    │ Replace with stacked   │ GPU: compile time   │
│                       │ 2D array + int indices │ TPU: runtime        │
├───────────────────────┼───────────────────────┼─────────────────────┤
│ MLE update per round  │ Already O(N) parallel; │ GPU: large win      │
│                       │ JIT-compile it         │ TPU: large win      │
├───────────────────────┼───────────────────────┼─────────────────────┤
│ Round loop (n rounds) │ Cannot parallelise —   │ N/A                 │
│                       │ each round needs prev  │                     │
│                       │ challenge              │                     │
├───────────────────────┼───────────────────────┼─────────────────────┤
│ jnp.sum for claim0    │ JAX reduces in         │ GPU: moderate       │
│                       │ parallel automatically │                     │
└───────────────────────┴───────────────────────┴─────────────────────┘
```

**The bottleneck is always the innermost elementwise ops** on the `2^n` table. That is where GPU/TPU wins big. The `n`-round outer loop is unavoidably sequential.

---
## Section 9 — Benchmarking Like the Assignment

The assignment asks you to run warmup rounds followed by timed runs. Here's how to structure that correctly with JAX — the key trap is that **JAX is asynchronous by default**: the Python call returns immediately and the GPU work happens in the background. Always call `.block_until_ready()` inside your timing window.

In [ ]:
import jax
import jax.numpy as jnp
import time


def benchmark_sumcheck(num_vars, expression, q_val, warmup=3, runs=8):
    """
    Benchmark the full sumcheck for a given problem size.

    num_vars: number of Boolean variables (table size = 2^num_vars)
    expression: list[list[str]]
    q_val: int — prime modulus
    warmup: number of warmup calls (triggers JIT compile on first call)
    runs: number of timed calls
    """
    N = 2 ** num_vars
    q = jnp.uint32(q_val)
    key = jax.random.PRNGKey(42)

    # Build random eval_tables
    var_names = list({v for term in expression for v in term})
    eval_tables = {
        var: jax.random.randint(key, (N,), 0, q_val, dtype=jnp.uint32)
        for var in var_names
    }

    # Random challenges (num_vars - 1 of them; last one is verifier-only)
    challenges = jax.random.randint(key, (num_vars - 1,), 1, q_val, dtype=jnp.uint32)

    expr_str = " + ".join("*".join(t) for t in expression)
    print(f"\n{'='*60}")
    print(f"Benchmarking: vars={num_vars}  expr='{expr_str}'  N=2^{num_vars}={N:,}")
    print(f"Backend: {jax.default_backend()}")

    # Warmup (also triggers JIT compilation on first call)
    print(f"Running {warmup} warmup calls...", end=" ")
    for i in range(warmup):
        claim0, round_evals = sumcheck_32(
            eval_tables, q=q, expression=expression,
            challenges=challenges, num_rounds=num_vars
        )
        round_evals.block_until_ready()  # wait for GPU to finish
    print("done.")

    # Timed runs
    times_ms = []
    for i in range(runs):
        t0 = time.perf_counter()
        claim0, round_evals = sumcheck_32(
            eval_tables, q=q, expression=expression,
            challenges=challenges, num_rounds=num_vars
        )
        round_evals.block_until_ready()  # MUST block to get accurate timing
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000)

    mean_ms  = sum(times_ms) / len(times_ms)
    min_ms   = min(times_ms)
    max_ms   = max(times_ms)

    print(f"Results over {runs} runs:")
    print(f"  Mean: {mean_ms:.2f} ms")
    print(f"  Min:  {min_ms:.2f} ms")
    print(f"  Max:  {max_ms:.2f} ms")
    print(f"  Per-round mean: {mean_ms/num_vars:.2f} ms")
    return mean_ms


Q = 4_294_967_291  # largest 32-bit prime

# Run the benchmark suite matching the assignment requirements
for num_vars in [4, 16]:
    for expr in [[['a']], [['a','b']], [['a','b'],['c']], [['a','b','c']]]:
        benchmark_sumcheck(num_vars, expr, Q, warmup=3, runs=8)

---
## Section 10 — Putting It All Together

Here is the clean, optimised version of the full implementation ready to drop into `student.py`.

Checklist before submitting:
- [ ] `mod_add_32`, `mod_sub_32`, `mod_mul_32` all use 64-bit intermediate promotion.
- [ ] `mle_update_32` uses those primitives.
- [ ] `sumcheck_32` returns `(claim0, round_evals)` where both are JAX arrays.
- [ ] `challenges` passed in has length `num_rounds - 1` (no final verifier challenge).
- [ ] `claim0` is computed **before** any MLE updates (it's the initial sum).
- [ ] Tables are updated in-place each round using the challenge.
- [ ] `round_evals` has shape `(num_rounds, degree+1)`.

In [ ]:
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)


# ── Primitives ────────────────────────────────────────────────────────────────

def mod_add_32(a, b, q):
    return ((a.astype(jnp.uint64) + b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)

def mod_sub_32(a, b, q):
    q64 = jnp.uint64(q)
    return ((a.astype(jnp.uint64) + q64 - b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

def mod_mul_32(a, b, q):
    return ((a.astype(jnp.uint64) * b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)


# ── MLE Update ───────────────────────────────────────────────────────────────

def mle_update_32(zero_eval, one_eval, target_eval, *, q):
    diff   = mod_sub_32(one_eval, zero_eval, q)
    scaled = mod_mul_32(diff, target_eval, q)
    return mod_add_32(scaled, zero_eval, q)


# ── Expression evaluation ─────────────────────────────────────────────────────

def eval_expression(tables, expression, q):
    """Evaluate composed polynomial pointwise. tables: dict[str->array]."""
    total = None
    for term in expression:
        term_val = tables[term[0]]
        for var in term[1:]:
            term_val = mod_mul_32(term_val, tables[var], q)
        total = term_val if total is None else mod_add_32(total, term_val, q)
    return total


# ── Full SumCheck ─────────────────────────────────────────────────────────────

def sumcheck_32(eval_tables, *, q, expression, challenges, num_rounds):
    degree          = sum(len(term) for term in expression)
    num_eval_points = degree + 1

    # claim0: initial sum over full table
    f_vals = eval_expression(eval_tables, expression, q)
    claim0 = (jnp.sum(f_vals.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)

    tables  = {k: v for k, v in eval_tables.items()}  # shallow copy
    all_round_evals = []

    for round_idx in range(num_rounds):
        evens = {var: tbl[0::2] for var, tbl in tables.items()}
        odds  = {var: tbl[1::2] for var, tbl in tables.items()}

        round_evals = []
        for t_int in range(num_eval_points):
            t = jnp.array(t_int, dtype=jnp.uint32)
            if t_int == 0:
                tables_t = evens
            elif t_int == 1:
                tables_t = odds
            else:
                tables_t = {var: mle_update_32(evens[var], odds[var], t, q=q) for var in tables}
            vals = eval_expression(tables_t, expression, q)
            g_t  = (jnp.sum(vals.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)
            round_evals.append(g_t)
        all_round_evals.append(jnp.stack(round_evals))

        # Apply MLE update with this round's challenge (skip if last round)
        if round_idx < len(challenges):
            r = challenges[round_idx]
            tables = {var: mle_update_32(evens[var], odds[var], r, q=q) for var in tables}

    round_evals_2d = jnp.stack(all_round_evals)  # shape (num_rounds, degree+1)
    return claim0, round_evals_2d


# ── Final verification ────────────────────────────────────────────────────────
print("Running final self-check on the 3-variable example from the intro doc...")
Q = jnp.uint32(17)
tables_check = {'a': jnp.array([0,1,2,3,4,5,6,7], dtype=jnp.uint32)}
chals_check  = jnp.array([8, 5], dtype=jnp.uint32)

c0, re = sumcheck_32(tables_check, q=Q, expression=[['a']],
                     challenges=chals_check, num_rounds=3)

expected = {
    'claim0': 11,
    'round_evals': [[12, 16], [3, 7], [1, 5]]
}

ok = (int(c0) == expected['claim0']) and (re.tolist() == expected['round_evals'])
print(f"  claim0:      {int(c0)}  (expected {expected['claim0']})")
print(f"  round_evals: {re.tolist()}")
print(f"  Overall: {'PASS' if ok else 'FAIL'}")

---
## Quick Reference

### API the harness calls
```python
claim0, round_evals = sumcheck(eval_tables, q=q, expression=expression,
                               challenges=challenges, num_rounds=num_rounds,
                               bit_width=32)
```
- `claim0`: scalar JAX uint32 — the initial sum S
- `round_evals`: JAX array shape `(num_rounds, degree+1)` — per-round polynomial evaluations

### Degree table
```
Expression       Degree   eval points   round_evals columns
─────────────────────────────────────────────────────────────
[['a']]             1        2           [g(0), g(1)]
[['a','b']]         2        3           [g(0), g(1), g(2)]
[['a','b'],['c']]   2        3           [g(0), g(1), g(2)]
[['a','b','c']]     3        4           [g(0), g(1), g(2), g(3)]
```

### MLE update
```python
mle_update(z, o, t) = (o - z) * t + z   (mod q)
#  z = even-index entry  (value when variable = 0)
#  o = odd-index  entry  (value when variable = 1)
#  t = random challenge to interpolate to
```

### GPU optimisation checklist
1. Wrap all fixed-shape functions with `@jax.jit`
2. Use `uint64` for intermediate computations to avoid overflow
3. Use `stacked_tables` (2D array) instead of dicts inside JIT
4. Always call `.block_until_ready()` when timing
5. The `n`-round outer loop is sequential — focus GPU effort on the table ops inside each round

### Useful links
- JAX docs: https://jax.readthedocs.io/
- JAX JIT guide: https://jax.readthedocs.io/en/latest/jit-compilation.html
- JAX vmap guide: https://jax.readthedocs.io/en/latest/automatic-vectorization.html
- NoCap paper (low-degree SumCheck): https://people.csail.mit.edu/devadas/pubs/micro24_nocap.pdf